In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from models.lstm_cnn_attention import LSTMCNNAttention

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [3]:
# Sanity check for LSTM-CNN-Attention model

model = LSTMCNNAttention()

x = torch.randn(8, 75, 3)  # batch of 8 samples
y = model(x)

print("Output shape:", y.shape)

Output shape: torch.Size([8, 3])


In [4]:
# Sanity check for Autoencoder 

from models.sequence_autoencoder import SequenceAutoencoder

ae = SequenceAutoencoder()

x = torch.randn(4, 75, 3)
recon = ae(x)

print("Input shape:", x.shape)
print("Reconstructed shape:", recon.shape)

Input shape: torch.Size([4, 75, 3])
Reconstructed shape: torch.Size([4, 75, 3])


In [5]:
# Load pre-generated dataset
X_train = np.load("../data/X_train.npy")
y_train = np.load("../data/y_train.npy")

X_val = np.load("../data/X_val.npy")
y_val = np.load("../data/y_val.npy")

print("Train shape:", X_train.shape, y_train.shape)
print("Val shape:", X_val.shape, y_val.shape)

Train shape: (10500, 75, 3) (10500,)
Val shape: (2250, 75, 3) (2250,)


In [6]:
# Convert to PyTorch tensors
'''
    Why?
    PyTorch models only work with tensors
    Labels must be long for classification
'''

X_train_t = torch.tensor(X_train, dtype = torch.float32)
y_train_t = torch.tensor(y_train, dtype = torch.long)

X_val_t = torch.tensor(X_val, dtype = torch.float32)
y_val_t = torch.tensor(y_val, dtype = torch.long)

In [7]:
# Model
model = LSTMCNNAttention()

# Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 1e-3)

# Training params
EPOCHS = 15
BATCH_SIZE = 64

Simple. No tuning yet

In [8]:
def train_one_epoch(model, X, y, optimizer, criterion, batch_size):

    model.train()
    total_loss = 0
    correct = 0
    num_batches = (len(X) + batch_size - 1) // batch_size

    for i in range(0, len(X), batch_size):
        xb = X[i:i + batch_size]
        yb = y[i:i + batch_size]

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = outputs.argmax(dim = 1)
        correct += (preds == yb).sum().item()

        # Print progress
        print(f"Batch {i//batch_size + 1}/{num_batches} - Loss: {loss.item():.4f}")

    acc = correct / len(X)
    return total_loss, acc

In [9]:
# Validation Loop

def evaluate(model, X, y, criterion):
    model.eval()
    total_loss = 0
    correct = 0

    with torch.no_grad():
        outputs = model(X)
        loss = criterion(outputs, y)

        total_loss = loss.item()
        preds = outputs.argmax(dim = 1)
        correct = (preds == y).sum().item()

    acc = correct / len(X)
    return total_loss, acc

In [10]:
# Train Model

for epoch in range(EPOCHS):
    
    train_loss, train_acc = train_one_epoch(
        model, X_train_t, y_train_t, optimizer, criterion, BATCH_SIZE
    )

    val_loss, val_acc = evaluate(
        model, X_val_t, y_val_t, criterion
    )

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Batch 1/165 - Loss: 1.0990
Batch 2/165 - Loss: 1.0990
Batch 3/165 - Loss: 1.0894
Batch 4/165 - Loss: 1.0835
Batch 5/165 - Loss: 1.0903
Batch 6/165 - Loss: 1.0863
Batch 7/165 - Loss: 1.0629
Batch 8/165 - Loss: 1.0568
Batch 9/165 - Loss: 1.0573
Batch 10/165 - Loss: 1.0500
Batch 11/165 - Loss: 1.0431
Batch 12/165 - Loss: 1.0287
Batch 13/165 - Loss: 1.0218
Batch 14/165 - Loss: 0.9915
Batch 15/165 - Loss: 0.9878
Batch 16/165 - Loss: 0.9915
Batch 17/165 - Loss: 0.9532
Batch 18/165 - Loss: 0.9264
Batch 19/165 - Loss: 0.9108
Batch 20/165 - Loss: 0.9043
Batch 21/165 - Loss: 0.8726
Batch 22/165 - Loss: 0.8442
Batch 23/165 - Loss: 0.8043
Batch 24/165 - Loss: 0.7701
Batch 25/165 - Loss: 0.7588
Batch 26/165 - Loss: 0.7556
Batch 27/165 - Loss: 0.7443
Batch 28/165 - Loss: 0.7426
Batch 29/165 - Loss: 0.6270
Batch 30/165 - Loss: 0.5729
Batch 31/165 - Loss: 0.6142
Batch 32/165 - Loss: 0.6624
Batch 33/165 - Loss: 0.5715
Batch 34/165 - Loss: 0.5377
Batch 35/165 - Loss: 0.5381
Batch 36/165 - Loss: 0.5439
B

```
Batch 1/165 - Loss: 1.0990
Batch 2/165 - Loss: 1.0990
Batch 3/165 - Loss: 1.0894
Batch 4/165 - Loss: 1.0835
Batch 5/165 - Loss: 1.0903
...
Batch 163/165 - Loss: 0.0109
Batch 164/165 - Loss: 0.0107
Batch 165/165 - Loss: 0.0208
```

Epoch 15/15 | Train Acc: 0.986 | Val Acc: 0.997

On synthetic data, you should see:

- accuracy quickly rise above 90%
- validation track training closely

That confirms:
- dataset is usable
- model is learning
- pipeline is correct

In [11]:
# Load test data
X_test = np.load("../data/X_test.npy")
y_test = np.load("../data/y_test.npy")

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print("Test shape:", X_test_t.shape, y_test_t.shape)

Test shape: torch.Size([2250, 75, 3]) torch.Size([2250])


In [12]:
# Evaluate on test set
test_loss, test_acc = evaluate(
    model, X_test_t, y_test_t, criterion
)

print(f"Test Accuracy: {test_acc:.3f}")

Test Accuracy: 0.998


You should expect:
- Test accuracy ≈ validation accuracy
- Slight drop is okay

In [13]:
# Confusion Matrix

from sklearn.metrics import confusion_matrix, classification_report

model.eval()
with torch.no_grad():
    outputs = model(X_test_t)
    preds = outputs.argmax(dim=1).cpu().numpy()

cm = confusion_matrix(y_test, preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(y_test, preds, target_names=[
    "Light Braking", "Normal Braking", "Emergency Braking"
]))

Confusion Matrix:
 [[775   4   0]
 [  0 711   0]
 [  0   0 760]]

Classification Report:
                   precision    recall  f1-score   support

    Light Braking       1.00      0.99      1.00       779
   Normal Braking       0.99      1.00      1.00       711
Emergency Braking       1.00      1.00      1.00       760

         accuracy                           1.00      2250
        macro avg       1.00      1.00      1.00      2250
     weighted avg       1.00      1.00      1.00      2250



In [14]:
# Save trained model
torch.save(model.state_dict(), "../models/lstm_cnn_attention_baseline.pth")
print("Model saved successfully.")

Model saved successfully.


Files after saving this model 

models/
- lstm_cnn_attention.py
- lstm_cnn_attention_baseline.pth


## Autoencoder

In [15]:
ae = SequenceAutoencoder()

ae_criterion = nn.MSELoss()
ae_optimizer = optim.Adam(ae.parameters(), lr=1e-3)

AE_EPOCHS = 20
AE_BATCH_SIZE = 64

In [16]:
# Train Autoencoder 
def train_autoencoder(model, X, optimizer, criterion, batch_size):
    model.train()
    total_loss = 0

    for i in range(0, len(X), batch_size):
        xb = X[i:i+batch_size]

        optimizer.zero_grad()
        recon = model(xb)
        loss = criterion(recon, xb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(X)

In [17]:
for epoch in range(AE_EPOCHS):
    loss = train_autoencoder(
        ae, X_train_t, ae_optimizer, ae_criterion, AE_BATCH_SIZE
    )

    print(f"AE Epoch {epoch+1}/{AE_EPOCHS} | Reconstruction Loss: {loss:.6f}")

AE Epoch 1/20 | Reconstruction Loss: 1.573429
AE Epoch 2/20 | Reconstruction Loss: 0.075039
AE Epoch 3/20 | Reconstruction Loss: 0.025374
AE Epoch 4/20 | Reconstruction Loss: 0.017062
AE Epoch 5/20 | Reconstruction Loss: 0.004242
AE Epoch 6/20 | Reconstruction Loss: 0.000468
AE Epoch 7/20 | Reconstruction Loss: 0.000388
AE Epoch 8/20 | Reconstruction Loss: 0.000373
AE Epoch 9/20 | Reconstruction Loss: 0.000358
AE Epoch 10/20 | Reconstruction Loss: 0.000344
AE Epoch 11/20 | Reconstruction Loss: 0.000329
AE Epoch 12/20 | Reconstruction Loss: 0.000314
AE Epoch 13/20 | Reconstruction Loss: 0.000300
AE Epoch 14/20 | Reconstruction Loss: 0.000285
AE Epoch 15/20 | Reconstruction Loss: 0.000271
AE Epoch 16/20 | Reconstruction Loss: 0.000258
AE Epoch 17/20 | Reconstruction Loss: 0.000245
AE Epoch 18/20 | Reconstruction Loss: 0.000233
AE Epoch 19/20 | Reconstruction Loss: 0.000221
AE Epoch 20/20 | Reconstruction Loss: 0.000210


In [18]:
# Save trained Autoencoder 
torch.save(ae.state_dict(), "../models/sequence_autoencoder.pth")

print("Autoencoder saved.")

Autoencoder saved.


In [19]:
# AE + Classifier Sanity check
from models.lstm_cnn_attention import AE_LSTMCNNAttention

ae_model = AE_LSTMCNNAttention()

x = torch.randn(4, 75, 3)
y = ae_model(x)

print("Output shape:", y.shape)

Output shape: torch.Size([4, 3])


In [20]:
# Train the integrated model
ae_classifier = AE_LSTMCNNAttention()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, ae_classifier.parameters()),
    lr = 1e-3
)

EPOCHS = 15
BATCH_SIZE = 64

In [21]:
for epoch in range(EPOCHS):
    
    train_loss, train_acc = train_one_epoch(
        ae_classifier, X_train_t, y_train_t,
        optimizer, criterion, BATCH_SIZE
    )

    val_loss, val_acc = evaluate(
        ae_classifier, X_val_t, y_val_t, criterion
    )

    print(
        f"[AE+CLS] Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Batch 1/165 - Loss: 1.0949
Batch 2/165 - Loss: 1.0915
Batch 3/165 - Loss: 1.0816
Batch 4/165 - Loss: 1.0733
Batch 5/165 - Loss: 1.0732
Batch 6/165 - Loss: 1.0664
Batch 7/165 - Loss: 1.0564
Batch 8/165 - Loss: 1.0419
Batch 9/165 - Loss: 1.0356
Batch 10/165 - Loss: 1.0304
Batch 11/165 - Loss: 1.0207
Batch 12/165 - Loss: 1.0050
Batch 13/165 - Loss: 0.9800
Batch 14/165 - Loss: 0.9962
Batch 15/165 - Loss: 0.9861
Batch 16/165 - Loss: 1.0028
Batch 17/165 - Loss: 0.9163
Batch 18/165 - Loss: 0.8868
Batch 19/165 - Loss: 0.9331
Batch 20/165 - Loss: 0.9758
Batch 21/165 - Loss: 0.9213
Batch 22/165 - Loss: 0.8956
Batch 23/165 - Loss: 0.8355
Batch 24/165 - Loss: 0.8229
Batch 25/165 - Loss: 0.8635
Batch 26/165 - Loss: 0.9134
Batch 27/165 - Loss: 0.8355
Batch 28/165 - Loss: 0.8105
Batch 29/165 - Loss: 0.7167
Batch 30/165 - Loss: 0.6326
Batch 31/165 - Loss: 0.7282
Batch 32/165 - Loss: 0.7847
Batch 33/165 - Loss: 0.6970
Batch 34/165 - Loss: 0.6386
Batch 35/165 - Loss: 0.6110
Batch 36/165 - Loss: 0.7785
B

```
Batch 1/165 - Loss: 1.0949
Batch 2/165 - Loss: 1.0915
Batch 3/165 - Loss: 1.0816
Batch 4/165 - Loss: 1.0733
Batch 5/165 - Loss: 1.0732
...
Batch 163/165 - Loss: 0.0027
Batch 164/165 - Loss: 0.0032
Batch 165/165 - Loss: 0.0004
```

[AE+CLS] Epoch 15/15 | Train Acc: 0.996 | Val Acc: 0.994

## Test-set evaluation for AE + Classifier

In [22]:
# Load test data
X_test = np.load("../data/X_test.npy")
y_test = np.load("../data/y_test.npy")

X_test_t = torch.tensor(X_test, dtype = torch.float32)
y_test_t = torch.tensor(y_test, dtype = torch.long)

print("Test set shape:", X_test_t.shape, y_test_t.shape)

Test set shape: torch.Size([2250, 75, 3]) torch.Size([2250])


In [23]:
test_loss, test_acc = evaluate(
    ae_classifier, X_test_t, y_test_t, criterion
)

print(f"[AE+CLS] Test Accuracy: {test_acc:.4f}")

[AE+CLS] Test Accuracy: 0.9929


In [24]:
# Confusion Matrix

from sklearn.metrics import confusion_matrix, classification_report

ae_classifier.eval()
with torch.no_grad():
    outputs = ae_classifier(X_test_t)
    preds = outputs.argmax(dim = 1).cpu().numpy()

cm = confusion_matrix(y_test, preds)
print("Confusion Matrix (AE+CLS):\n", cm)

print("\nClassification Report (AE+CLS):")
print(classification_report(
    y_test,
    preds,
    target_names = ["Light Braking", "Normal Braking", "Emergency Braking"]
))

Confusion Matrix (AE+CLS):
 [[779   0   0]
 [ 16 695   0]
 [  0   0 760]]

Classification Report (AE+CLS):
                   precision    recall  f1-score   support

    Light Braking       0.98      1.00      0.99       779
   Normal Braking       1.00      0.98      0.99       711
Emergency Braking       1.00      1.00      1.00       760

         accuracy                           0.99      2250
        macro avg       0.99      0.99      0.99      2250
     weighted avg       0.99      0.99      0.99      2250



---
---

Test results:

Test Accuracy: 99.29%

Confusion Matrix:

    Light     → almost perfect
    Normal    → small confusion with Light (16 samples)
    Emergency → perfect

This tells us three important things:

✅ (1) No train–test leakage

If there was leakage:
- test accuracy would be ~100%
- confusion matrix would be perfectly diagonal

But we do have:
- small, realistic confusion (Normal ↔ Light)
- slightly lower test accuracy than val

This is healthy.

✅ (2) Emergency braking is learned robustly

This is critical for both real-world relevance & research credibility

Emergency braking:
- Precision = 1.00
- Recall = 1.00

This means the model has learned clear temporal patterns for emergency braking, not just thresholds.

⚠️ (3) The data distribution is still “easy”

Your concern was:

“The model has learned the data instead of understanding patterns.”

The correct refined statement is:

“The model understands patterns very well — but the patterns themselves are still too clean and consistent.”

This is a data realism issue, not a model issue. That’s an important distinction.

2️⃣ So… is this a problem?
❌ No, this is NOT a problem at this stage
✅ This is actually the expected outcome

Why?
- You deliberately started with clean synthetic data & controlled distributions
- This is baseline + first innovation validation

In real ML workflows:
- Validate architecture correctness ✅ (done)
- Validate training pipeline correctness ✅ (done)
- Validate controlled generalization ✅ (done)
- Then stress-test realism ❗ (next step)

---

## Adding noise and domain shift to test data

1. Sensor noise (Gaussian noise)

    Simulates:
    - speed sensor jitter
    - acceleration noise
    - pedal sensor imperfections

2. Brake pedal delay

    Simulates:
    - human reaction delay
    - actuator lag

3. Feature scaling drift

    Simulates:
    - calibration differences
    - different vehicles / drivers

In [25]:
# Create noisy / shifted test data
def add_sensor_noise(X, noise_std = 0.05):

    noise = np.random.normal(0, noise_std, X.shape)
    return X + noise


def add_brake_delay(X, delay_steps = 3):

    X_delayed = X.copy()
    X_delayed[:, delay_steps:, 2] = X[:, :-delay_steps, 2]
    X_delayed[:, :delay_steps, 2] = 0.0
    return X_delayed


def apply_feature_drift(X, scale_range = (0.9, 1.1)):

    scales = np.random.uniform(
        scale_range[0],
        scale_range[1],
        size = (1, 1, X.shape[2])
    )
    return X * scales

In [26]:
# Copy original test data
X_test_stress = X_test.copy()

# Apply noise & domain shift
X_test_stress = add_sensor_noise(X_test_stress, noise_std=0.08)
X_test_stress = add_brake_delay(X_test_stress, delay_steps=4)
X_test_stress = apply_feature_drift(X_test_stress)

# Convert to tensor
X_test_stress_t = torch.tensor(
    X_test_stress,
    dtype = torch.float32
)

print("Stressed test set created.")

Stressed test set created.


In [27]:
# Evaluate AE + CLS on stressed test set
stress_loss, stress_acc = evaluate(
    ae_classifier,
    X_test_stress_t,
    y_test_t,
    criterion
)

print(f"[AE+CLS] Stress Test Accuracy: {stress_acc:.4f}")

[AE+CLS] Stress Test Accuracy: 0.9933


In [28]:
# Confusion Matrix
ae_classifier.eval()
with torch.no_grad():
    outputs = ae_classifier(X_test_stress_t)
    preds = outputs.argmax(dim = 1).cpu().numpy()

cm_stress = confusion_matrix(y_test, preds)

print("Confusion Matrix (AE+CLS – Stressed):\n", cm_stress)

print("\nClassification Report (AE+CLS – Stressed):")
print(classification_report(
    y_test,
    preds,
    target_names = ["Light Braking", "Normal Braking", "Emergency Braking"]
))


Confusion Matrix (AE+CLS – Stressed):
 [[779   0   0]
 [  5 696  10]
 [  0   0 760]]

Classification Report (AE+CLS – Stressed):
                   precision    recall  f1-score   support

    Light Braking       0.99      1.00      1.00       779
   Normal Braking       1.00      0.98      0.99       711
Emergency Braking       0.99      1.00      0.99       760

         accuracy                           0.99      2250
        macro avg       0.99      0.99      0.99      2250
     weighted avg       0.99      0.99      0.99      2250



In [29]:
# Compare with baseline
baseline_loss, baseline_stress_acc = evaluate(
    model,  # baseline LSTM-CNN-Attention
    X_test_stress_t,
    y_test_t,
    criterion
)

print(f"[Baseline] Stress Test Accuracy: {baseline_stress_acc:.4f}")

[Baseline] Stress Test Accuracy: 0.9924


#### Why didn’t accuracy drop meaningfully?

The labels are still trivially recoverable even after noise & shift.

The synthetic data has very strong label–feature coupling, likely something like:
- Light braking → low brake pedal, mild decel
- Normal braking → moderate, smooth patterns
- Emergency braking → very strong, sustained signals

Even after:
- Gaussian noise
- Small delays
- Feature scaling

#### Why the stress test didn’t truly stress the model

❌ What we changed
- Added noise
- Shifted brake pedal
- Scaled features

❌ What we did NOT change (this is the problem)
- Class overlap
- Ambiguous braking events
- Mixed braking styles
- Temporal inconsistency
- Label uncertainty

In real driving:
- Light vs Normal braking often overlap
- Emergency braking is not always “max pedal”
- Drivers brake inconsistently
- Signals contradict each other

The synthetic generator currently never produces ambiguity.

---
How real research papers avoid this trap

Good ML-for-control papers do one (or more) of the following:
1. Predict future intention (harder)
2. Introduce class overlap
3. Use soft / probabilistic labels
4. Mix braking modes in one window
5. Train on one distribution, test on another

We currently have:
- Same generator
- Same logic
- Same label rules

    → even noisy data still follows the same rules

---

Now we'll make the task genuinely harder

→ Ambiguous + overlapping braking data

This means:
- Light & Normal braking overlap intentionally
- Emergency braking sometimes looks “normal” at first
- Label is based on future behavior, not current

This is the right scientific fix.

- Makes accuracy drop meaningfully
- Tests temporal understanding

---

## Loading hard dataset

In [30]:
# Load HARD dataset
X_train_h = np.load("../data/X_train_hard.npy")
y_train_h = np.load("../data/y_train_hard.npy")

X_val_h = np.load("../data/X_val_hard.npy")
y_val_h = np.load("../data/y_val_hard.npy")

X_test_h = np.load("../data/X_test_hard.npy")
y_test_h = np.load("../data/y_test_hard.npy")

print("Hard train:", X_train_h.shape, y_train_h.shape)
print("Hard val:", X_val_h.shape, y_val_h.shape)
print("Hard test:", X_test_h.shape, y_test_h.shape)

Hard train: (10500, 75, 3) (10500,)
Hard val: (2250, 75, 3) (2250,)
Hard test: (2250, 75, 3) (2250,)


In [31]:
# Convert to PyTorch tensors
X_train_h_t = torch.tensor(X_train_h, dtype = torch.float32)
y_train_h_t = torch.tensor(y_train_h, dtype = torch.long)

X_val_h_t = torch.tensor(X_val_h, dtype = torch.float32)
y_val_h_t = torch.tensor(y_val_h, dtype = torch.long)

X_test_h_t = torch.tensor(X_test_h, dtype = torch.float32)
y_test_h_t = torch.tensor(y_test_h, dtype = torch.long)

In [32]:
# Initialize a FRESH baseline model
baseline_hard = LSTMCNNAttention()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_hard.parameters(), lr = 1e-3)

EPOCHS = 20
BATCH_SIZE = 64

In [33]:
# Train baseline on HARD data
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        baseline_hard,
        X_train_h_t,
        y_train_h_t,
        optimizer,
        criterion,
        BATCH_SIZE
    )

    val_loss, val_acc = evaluate(
        baseline_hard,
        X_val_h_t,
        y_val_h_t,
        criterion
    )

    print(
        f"[Baseline-HARD] Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Batch 1/165 - Loss: 1.1109
Batch 2/165 - Loss: 1.0942
Batch 3/165 - Loss: 1.0847
Batch 4/165 - Loss: 1.1155
Batch 5/165 - Loss: 1.1121
Batch 6/165 - Loss: 1.0838
Batch 7/165 - Loss: 1.1038
Batch 8/165 - Loss: 1.1008
Batch 9/165 - Loss: 1.0931
Batch 10/165 - Loss: 1.0917
Batch 11/165 - Loss: 1.0902
Batch 12/165 - Loss: 1.0924
Batch 13/165 - Loss: 1.0910
Batch 14/165 - Loss: 1.0817
Batch 15/165 - Loss: 1.0855
Batch 16/165 - Loss: 1.1006
Batch 17/165 - Loss: 1.1133
Batch 18/165 - Loss: 1.0818
Batch 19/165 - Loss: 1.1030
Batch 20/165 - Loss: 1.0945
Batch 21/165 - Loss: 1.0848
Batch 22/165 - Loss: 1.1070
Batch 23/165 - Loss: 1.0797
Batch 24/165 - Loss: 1.1068
Batch 25/165 - Loss: 1.1273
Batch 26/165 - Loss: 1.0940
Batch 27/165 - Loss: 1.1046
Batch 28/165 - Loss: 1.0657
Batch 29/165 - Loss: 1.0800
Batch 30/165 - Loss: 1.0501
Batch 31/165 - Loss: 1.0808
Batch 32/165 - Loss: 1.0552
Batch 33/165 - Loss: 1.0865
Batch 34/165 - Loss: 1.0577
Batch 35/165 - Loss: 1.0649
Batch 36/165 - Loss: 1.0480
B

```
Batch 1/165 - Loss: 1.1109
Batch 2/165 - Loss: 1.0942
Batch 3/165 - Loss: 1.0847
Batch 4/165 - Loss: 1.1155
Batch 5/165 - Loss: 1.1121
...
Batch 163/165 - Loss: 0.5760
Batch 164/165 - Loss: 0.5711
Batch 165/165 - Loss: 0.1820
```

[Baseline-HARD] Epoch 20/20 | Train Acc: 0.712 | Val Acc: 0.719

In [34]:
# Evaluate baseline on HARD test set
test_loss_h, test_acc_h = evaluate(
    baseline_hard,
    X_test_h_t,
    y_test_h_t,
    criterion
)

print(f"[Baseline-HARD] Test Accuracy: {test_acc_h:.4f}")

[Baseline-HARD] Test Accuracy: 0.6956


In [35]:
# Confusion matrix
baseline_hard.eval()
with torch.no_grad():
    outputs = baseline_hard(X_test_h_t)
    preds = outputs.argmax(dim=1).cpu().numpy()

cm_hard = confusion_matrix(y_test_h, preds)

print("Confusion Matrix (Baseline-HARD):\n", cm_hard)

print("\nClassification Report (Baseline-HARD):")
print(classification_report(
    y_test_h,
    preds,
    target_names = ["Light Braking", "Normal Braking", "Emergency Braking"]
))

Confusion Matrix (Baseline-HARD):
 [[598 193   3]
 [217 487 102]
 [ 12 158 480]]

Classification Report (Baseline-HARD):
                   precision    recall  f1-score   support

    Light Braking       0.72      0.75      0.74       794
   Normal Braking       0.58      0.60      0.59       806
Emergency Braking       0.82      0.74      0.78       650

         accuracy                           0.70      2250
        macro avg       0.71      0.70      0.70      2250
     weighted avg       0.70      0.70      0.70      2250



Confusion matrix interpretation:

- Light ↔ Normal: heavy confusion ✅
- Emergency: still relatively strong ✅
- No class collapse ✅


Analyzing class-wise behavior:

1. Light Braking
- Precision: 0.72
- Recall: 0.75

    → Reasonable, but confused with Normal (expected)

2. Normal Braking (hardest class)
- Precision: 0.58
- Recall: 0.60

    → This is exactly what real data looks like
    → Normal braking sits between light and emergency

3. Emergency Braking
- Precision: 0.82
- Recall: 0.74

    → Still learned reasonably well
    → Good sign for safety-critical behavior

Baseline-HARD results (~69.6%)

---

## Training AE+CLS on HARD dataset

In [36]:
from models.lstm_cnn_attention import AE_LSTMCNNAttention

ae_cls_hard = AE_LSTMCNNAttention(latent_dim = 4)

criterion = nn.CrossEntropyLoss()

'''
    Encoder is frozen
    Only CNN, LSTM, Attention, FC train
'''
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, ae_cls_hard.parameters()),
    lr = 1e-3
)

EPOCHS = 20
BATCH_SIZE = 64

In [37]:
# Train AE+CLS on HARD data
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        ae_cls_hard,
        X_train_h_t,
        y_train_h_t,
        optimizer,
        criterion,
        BATCH_SIZE
    )

    val_loss, val_acc = evaluate(
        ae_cls_hard,
        X_val_h_t,
        y_val_h_t,
        criterion
    )

    print(
        f"[AE+CLS-HARD] Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Batch 1/165 - Loss: 1.1026
Batch 2/165 - Loss: 1.1007
Batch 3/165 - Loss: 1.1172
Batch 4/165 - Loss: 1.0882
Batch 5/165 - Loss: 1.0946
Batch 6/165 - Loss: 1.0985
Batch 7/165 - Loss: 1.0862
Batch 8/165 - Loss: 1.0907
Batch 9/165 - Loss: 1.0983
Batch 10/165 - Loss: 1.0885
Batch 11/165 - Loss: 1.0907
Batch 12/165 - Loss: 1.0920
Batch 13/165 - Loss: 1.0914
Batch 14/165 - Loss: 1.0759
Batch 15/165 - Loss: 1.0865
Batch 16/165 - Loss: 1.0949
Batch 17/165 - Loss: 1.0971
Batch 18/165 - Loss: 1.0781
Batch 19/165 - Loss: 1.0896
Batch 20/165 - Loss: 1.0706
Batch 21/165 - Loss: 1.0668
Batch 22/165 - Loss: 1.0931
Batch 23/165 - Loss: 1.0691
Batch 24/165 - Loss: 1.0917
Batch 25/165 - Loss: 1.0986
Batch 26/165 - Loss: 1.0711
Batch 27/165 - Loss: 1.0634
Batch 28/165 - Loss: 1.0470
Batch 29/165 - Loss: 1.0464
Batch 30/165 - Loss: 1.0331
Batch 31/165 - Loss: 1.0418
Batch 32/165 - Loss: 1.0112
Batch 33/165 - Loss: 1.0183
Batch 34/165 - Loss: 0.9941
Batch 35/165 - Loss: 1.0003
Batch 36/165 - Loss: 0.9806
B

```
Batch 1/165 - Loss: 1.1026
Batch 2/165 - Loss: 1.1007
Batch 3/165 - Loss: 1.1172
Batch 4/165 - Loss: 1.0882
Batch 5/165 - Loss: 1.0946
...
Batch 163/165 - Loss: 0.5834
Batch 164/165 - Loss: 0.5618
Batch 165/165 - Loss: 0.1900
```

[AE+CLS-HARD] Epoch 20/20 | Train Acc: 0.709 | Val Acc: 0.646

In [38]:
# Evaluate AE+CLS on HARD test set
test_loss_h_ae, test_acc_h_ae = evaluate(
    ae_cls_hard,
    X_test_h_t,
    y_test_h_t,
    criterion
)

print(f"[AE+CLS-HARD] Test Accuracy: {test_acc_h_ae:.4f}")

[AE+CLS-HARD] Test Accuracy: 0.6409


In [39]:
# Confusion Matrix
ae_cls_hard.eval()
with torch.no_grad():
    outputs = ae_cls_hard(X_test_h_t)
    preds = outputs.argmax(dim = 1).cpu().numpy()

cm_hard_ae = confusion_matrix(y_test_h, preds)

print("Confusion Matrix (AE+CLS-HARD):\n", cm_hard_ae)

print("\nClassification Report (AE+CLS-HARD):")
print(classification_report(
    y_test_h,
    preds,
    target_names = ["Light Braking", "Normal Braking", "Emergency Braking"]
))

Confusion Matrix (AE+CLS-HARD):
 [[376 393  25]
 [ 77 502 227]
 [  2  84 564]]

Classification Report (AE+CLS-HARD):
                   precision    recall  f1-score   support

    Light Braking       0.83      0.47      0.60       794
   Normal Braking       0.51      0.62      0.56       806
Emergency Braking       0.69      0.87      0.77       650

         accuracy                           0.64      2250
        macro avg       0.68      0.65      0.64      2250
     weighted avg       0.67      0.64      0.64      2250



#### HARD dataset results

Model	            Test Accuracy
Baseline (HARD)	    0.6956
AE + CLS (HARD)	    0.6409

So yes — AE+CLS performs worse than baseline on HARD data.

This is not a failure. It's a meaningful scientific outcome.

Key observation
- The autoencoder helped on easy data
- The autoencoder hurt on ambiguous HARD data

Why? Because of one design choice we made on purpose:

🔒 We froze the encoder

That means:
- The encoder learned representations optimized for reconstruction
- Not optimized for discriminating subtle class boundaries
- On HARD data, discrimination matters more than denoising

So the AE is currently acting like a constraint, not a helper.


Confusion matrix tells the full story

AE+CLS-HARD confusion highlights

🔴 Light Braking
- Recall dropped to 0.47
- Many light samples pushed into Normal
- Means: encoder compressed away subtle differences

🟡 Normal Braking
- Recall 0.62 (slightly better than Light)
- Still very ambiguous (expected)

🟢 Emergency Braking
- Recall 0.87 → best-performing class
- This is actually very good
- AE preserved strong, salient signals

Interpretation:

The autoencoder preserves high-energy / dominant patterns well (emergency braking), but suppresses fine-grained distinctions (light vs normal).

---
We've now demonstrated three important things:

1. Data difficulty matters more than model complexity
2. Representation learning is not universally beneficial
3. Frozen encoders can harm discriminative tasks under ambiguity

---

In [40]:
# Sanity check (after unfreezing last layer of AE)

model = AE_LSTMCNNAttention()

for name, param in model.named_parameters():
    if "autoencoder.encoder" in name:
        print(name, param.requires_grad)


autoencoder.encoder.0.weight False
autoencoder.encoder.0.bias False
autoencoder.encoder.2.weight True
autoencoder.encoder.2.bias True


In [41]:
# Initialize fine-tuned AE+CLS
from models.lstm_cnn_attention import AE_LSTMCNNAttention

ae_cls_ft = AE_LSTMCNNAttention(latent_dim=4)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, ae_cls_ft.parameters()),
    lr = 5e-4  # smaller LR for fine-tuning
)

EPOCHS = 20
BATCH_SIZE = 64

In [42]:
# Train on HARD dataset (fine-tuning)
for epoch in range(EPOCHS):
    train_loss, train_acc = train_one_epoch(
        ae_cls_ft,
        X_train_h_t,
        y_train_h_t,
        optimizer,
        criterion,
        BATCH_SIZE
    )

    val_loss, val_acc = evaluate(
        ae_cls_ft,
        X_val_h_t,
        y_val_h_t,
        criterion
    )

    print(
        f"[AE+CLS-FT] Epoch {epoch+1}/{EPOCHS} | "
        f"Train Acc: {train_acc:.3f} | "
        f"Val Acc: {val_acc:.3f}"
    )

Batch 1/165 - Loss: 1.0906
Batch 2/165 - Loss: 1.1075
Batch 3/165 - Loss: 1.0911
Batch 4/165 - Loss: 1.0893
Batch 5/165 - Loss: 1.1216
Batch 6/165 - Loss: 1.0866
Batch 7/165 - Loss: 1.0906
Batch 8/165 - Loss: 1.0974
Batch 9/165 - Loss: 1.0888
Batch 10/165 - Loss: 1.0865
Batch 11/165 - Loss: 1.0963
Batch 12/165 - Loss: 1.0880
Batch 13/165 - Loss: 1.0943
Batch 14/165 - Loss: 1.0943
Batch 15/165 - Loss: 1.0820
Batch 16/165 - Loss: 1.1053
Batch 17/165 - Loss: 1.0990
Batch 18/165 - Loss: 1.0799
Batch 19/165 - Loss: 1.1000
Batch 20/165 - Loss: 1.1041
Batch 21/165 - Loss: 1.0919
Batch 22/165 - Loss: 1.0892
Batch 23/165 - Loss: 1.0761
Batch 24/165 - Loss: 1.1249
Batch 25/165 - Loss: 1.1078
Batch 26/165 - Loss: 1.1011
Batch 27/165 - Loss: 1.1010
Batch 28/165 - Loss: 1.0734
Batch 29/165 - Loss: 1.0832
Batch 30/165 - Loss: 1.0854
Batch 31/165 - Loss: 1.0995
Batch 32/165 - Loss: 1.0989
Batch 33/165 - Loss: 1.1047
Batch 34/165 - Loss: 1.0997
Batch 35/165 - Loss: 1.0938
Batch 36/165 - Loss: 1.0887
B

```
Batch 1/165 - Loss: 1.1064
Batch 2/165 - Loss: 1.0954
Batch 3/165 - Loss: 1.0955
Batch 4/165 - Loss: 1.1096
Batch 5/165 - Loss: 1.0996
...
Batch 163/165 - Loss: 0.5881
Batch 164/165 - Loss: 0.5908
Batch 165/165 - Loss: 0.1494
```

[AE+CLS-FT] Epoch 20/20 | Train Acc: 0.708 | Val Acc: 0.606

In [44]:
# Evaluate on HARD test set
test_loss_ft, test_acc_ft = evaluate(
    ae_cls_ft,
    X_test_h_t,
    y_test_h_t,
    criterion
)

print(f"[AE+CLS-FT] Test Accuracy: {test_acc_ft:.4f}")

[AE+CLS-FT] Test Accuracy: 0.6133


In [45]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report

ae_cls_ft.eval()
with torch.no_grad():
    outputs = ae_cls_ft(X_test_h_t)
    preds = outputs.argmax(dim = 1).cpu().numpy()

cm_ft = confusion_matrix(y_test_h, preds)

print("Confusion Matrix (AE+CLS Fine-Tuned):\n", cm_ft)

print("\nClassification Report (AE+CLS Fine-Tuned):")
print(classification_report(
    y_test_h,
    preds,
    target_names=["Light Braking", "Normal Braking", "Emergency Braking"]
))

Confusion Matrix (AE+CLS Fine-Tuned):
 [[308 456  30]
 [ 67 499 240]
 [  3  74 573]]

Classification Report (AE+CLS Fine-Tuned):
                   precision    recall  f1-score   support

    Light Braking       0.81      0.39      0.53       794
   Normal Braking       0.48      0.62      0.54       806
Emergency Braking       0.68      0.88      0.77       650

         accuracy                           0.61      2250
        macro avg       0.66      0.63      0.61      2250
     weighted avg       0.66      0.61      0.60      2250



## Autoencoder Experiment: Expectations vs Outcome

### Motivation
The autoencoder was introduced with the expectation that learning a compressed latent representation of braking signals would:
- Remove sensor noise
- Improve robustness
- Help generalization under ambiguous braking scenarios

This approach is commonly effective when noise dominates the signal and class boundaries remain well separated.

---

### Experimental Setup
We evaluated three configurations on the **HARD ambiguous braking dataset**, where:
- Braking intention labels depend on future behavior
- Feature distributions overlap significantly
- Temporal patterns are inconsistent and mixed

Models evaluated:
1. Baseline LSTM–CNN–Attention
2. Autoencoder + Classifier (encoder frozen)
3. Autoencoder + Classifier (partial fine-tuning of encoder)

---

### Results Summary

| Model | Test Accuracy |
|------|---------------|
| Baseline (HARD) | **~69.6%** |
| AE + Classifier (Frozen) | ~64.1% |
| AE + Classifier (Fine-tuned) | ~60.5% |

Key observations from confusion matrices:
- Emergency braking remained relatively robust
- Light vs Normal braking confusion increased significantly
- Fine-grained distinctions were consistently degraded by the autoencoder

---

### Analysis: Why the Autoencoder Failed Here
The sequence autoencoder was optimized for **reconstruction**, not **discrimination**.  
Under ambiguous braking conditions:
- Fine temporal variations are crucial for distinguishing classes
- The autoencoder compressed and smoothed these subtle cues
- This reduced class separability, especially for Light and Normal braking

Partial fine-tuning did not recover performance, as the encoder’s inductive bias remained misaligned with the classification objective.

---

### Conclusion
Autoencoder-based representation learning is **not suitable** for highly ambiguous, future-dependent braking intention prediction, where preserving fine-grained temporal details is more important than denoising.

This negative result is informative and motivates a shift toward **task-aligned supervision**, rather than further representation compression.

---

### Next Direction
Based on these findings, we pivot to **multitask learning**, where the model is jointly trained to:
- Predict braking intention (classification)
- Predict braking intensity (regression)

This approach provides richer supervision without suppressing discriminative temporal information.


## Genetic Algorithm Hyperparameter Optimization

This notebook focuses on baseline single-task and autoencoder-based models, but the final braking intention system also uses a **Genetic Algorithm (GA)** to tune hyperparameters for the multitask LSTM–CNN–Attention model.

### Why use a Genetic Algorithm here?

The braking intention model has a **non-convex, highly coupled hyperparameter space**:
- Learning rate interacts with batch size and optimizer dynamics
- LSTM hidden size and number of layers trade off capacity vs. overfitting
- Dropout and CNN filters influence both regularization and feature richness
- The multitask loss couples classification and regression behavior

Standard grid or random search can miss good regions when:
- The search space is mixed discrete/continuous
- There are strong **interactions between hyperparameters**
- The objective landscape is rugged with many local optima

A **Genetic Algorithm** is well-suited because it:
- Explores multiple regions of the space in parallel (population-based)
- Uses crossover to combine promising hyperparameter configurations
- Uses mutation to escape local optima and explore new areas
- Treats the model as a black box, requiring only a scalar fitness (validation F1)

### Hyperparameter Search Space (Multitask LSTM–CNN–Attention)

The GA in `models/genetic_algorithm_optimizer.py` searches over the following space:

| Hyperparameter        | Type / Range                | Notes |
|-----------------------|----------------------------|-------|
| `learning_rate`       | Continuous \[1e-4, 1e-2] (log-uniform) | Adam optimizer step size |
| `batch_size`          | Categorical {16, 32, 64, 128} | Trade-off between stability and speed |
| `lstm_hidden_size`    | Categorical {64, 128, 256} | LSTM capacity for temporal patterns |
| `num_lstm_layers`     | Categorical {1, 2, 3}      | Depth of temporal modeling |
| `dropout_rate`        | Continuous \[0.1, 0.5]     | Applied between stacked LSTM layers |
| `cnn_filters`         | Categorical {32, 64, 128}  | Width of temporal CNN feature maps |

The **fitness function** is the **validation macro F1 score** of the braking intention classifier on the multitask HARD dataset. This aligns the search directly with the safety-critical classification performance you care about.